# A/B Test Analysis: Does a New Landing Page Increase Conversions?

**Question:** Does the new landing page (B) give a higher conversion rate than the old page (A)?

**Data:** [A/B testing dataset on Kaggle](https://www.kaggle.com/datasets/zhangluyuan/ab-testing), included in this repository as `ab_data.csv`.

**Plan:**
1. Load and clean the data
2. Baseline: compare the raw conversion rates
3. Two-proportion z-test and 95% confidence interval
4. Sample-size calculation
5. Checks on the method: A/A test, peeking and stability
6. Sanity checks
7. Conclusion

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm

# Settings
DATA_PATH = "ab_data.csv"
ALPHA = 0.05      # significance level: we accept a 5% chance of a false positive
POWER = 0.80      # 80% chance of detecting a real effect of MIN_LIFT
MIN_LIFT = 0.005  # smallest useful lift: 0.5 percentage points (e.g. 12.0% -> 12.5%)
SEED = 42         # fixed random seed so every run gives the same numbers

## 1. Load and clean the data

Two kinds of rows would make the comparison unfair:
- **Mismatched rows:** a `control` user who saw the new page, or a `treatment` user who saw the old page. We can't tell which version they really had.
- **Duplicate users:** each user must be counted once, otherwise the observations are not independent. We keep the first visit.

In [ ]:
def clean_data(df):
    """Remove mismatched rows and duplicate users. Returns (clean data, report)."""
    report = {"rows_raw": len(df)}

    matched = (
        ((df["group"] == "control") & (df["landing_page"] == "old_page"))
        | ((df["group"] == "treatment") & (df["landing_page"] == "new_page"))
    )
    report["rows_mismatched"] = int((~matched).sum())
    df = df[matched]

    duplicated = df["user_id"].duplicated(keep="first")
    report["duplicate_users_removed"] = int(duplicated.sum())
    df = df[~duplicated].reset_index(drop=True)

    report["rows_clean"] = len(df)
    return df, report


raw = pd.read_csv(DATA_PATH, parse_dates=["timestamp"])
df, report = clean_data(raw)
report

## 2. Baseline: compare the raw conversion rates

The simplest possible answer: which group converted more often? The error bars show the 95% range for each rate.

In [ ]:
summary = df.groupby("group")["converted"].agg(users="count", conversions="sum", rate="mean")
summary

In [ ]:
rates = summary["rate"]
errors = 1.96 * np.sqrt(rates * (1 - rates) / summary["users"])  # 95% CI half-width

fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(["Control (old page)", "Treatment (new page)"], rates,
       yerr=errors, capsize=8, color=["#7f8c8d", "#2e86c1"])
ax.set_ylabel("Conversion rate")
ax.set_ylim(0.10, 0.13)  # zoomed in so the small difference is visible
ax.set_title("Conversion rate by group (95% CI)")
plt.show()